## Kaggle Harness Template\n\nThis notebook shows the minimal workflow:\n1) install deps + install the repo as a package\n2) run training/inference/submission via `kaggle-harness`\n\nTo use for a real competition, copy this notebook and change the `REPO_SRC` and the Hydra overrides.

In [ ]:
from pathlib import Path
import shutil

# If you add this repo to Kaggle as a Dataset, it will appear under /kaggle/input/<dataset_slug>/...
# Point REPO_SRC at the directory that contains requirements.txt and pyproject.toml
REPO_SRC = Path("/kaggle/input/your-repo-dataset/KaggleCompetition")

# Work from /kaggle/working (writable)
REPO = Path("/kaggle/working/KaggleCompetition")
if not REPO.exists():
    shutil.copytree(REPO_SRC, REPO)

%cd /kaggle/working/KaggleCompetition

!pip -q install -r requirements.txt
!pip -q install -e .

!kaggle-harness --version

In [ ]:
# Example: run the built-in template competition.
# For a real Kaggle competition, create a new module under competitions/<your_comp>/run.py
# and point --competition at it.

KAGGLE_COMP_SLUG = "your-competition-slug"  # e.g. "playground-series-s6e2"
DATA_DIR = f"/kaggle/input/{KAGGLE_COMP_SLUG}"

!kaggle-harness run \
  --competition competitions._template.run \
  --action train \
  --override competition.slug={KAGGLE_COMP_SLUG} \
  --override competition.data_dir={DATA_DIR} \
  --override competition.id_col=id \
  --override competition.target_col=target \
  --override competition.problem_type=classification \
  --override competition.metric=roc_auc

In [ ]:
# Optional: generate a NIM-based research brief from an existing competition folder.
# Requires setting NIM_API_KEY as a Kaggle Secret and exposing it as an environment variable.

!kaggle-harness research \
  --competition-dir PredictingHeartDisease \
  --slug playground-series-s6e2 \
  --repo-root .

In [ ]:
# Copy the newest submission to /kaggle/working/submission.csv for easy Kaggle submission UI upload
from pathlib import Path

runs_dir = Path("/kaggle/working").glob("runs/*/*/submissions/submission.csv")
runs = sorted(list(runs_dir), key=lambda p: p.stat().st_mtime, reverse=True)
latest = runs[0] if runs else None
print("Latest:", latest)
if latest:
    out = Path("/kaggle/working/submission.csv")
    out.write_bytes(latest.read_bytes())
    print("Wrote:", out)